# Code Constructions

Build HGP, QLP, BPC, LCS, BB, and Mitten codes and verify their CSS logicals.

In [1]:
import numpy as np
import stim

from quits.circuit import check_overlapping_CX
from quits import ErrorModel, CircuitBuildOptions
from quits.decoder import (
    sliding_window_bposd_circuit_mem,
    sliding_window_bposd_phenom_mem,
    sliding_window_bplsd_phenom_mem,
)
from quits.gf2_util import MultTableGroup
from quits.qldpc_code import BbCode, BpcCode, HgpCode, LcsCode, MittenCode, QlpCode
from quits.qldpc_code.mitten import sample_mitten_base_matrices
from quits.simulation import get_stim_mem_result


<a id='hgp-code'></a>

### Hypergraph product (HGP) code parameters

Reference: Tillich & Zemor, arXiv:0903.0566

- `h`: parity check matrix of the classical LDPC code (loaded from `../parity_check_matrices`)
- `HgpCode(h, h)`: constructs the HGP code from two classical parity check matrices
- `seed`: used by `build_circuit` to assign edge directions and colors

The parity check matrix `h` can be found from `quits.ldpc_utility.generate_ldpc`, following the method in [A. Grospellier, *Constant time decoding of quantum expander codes and application to fault-tolerant quantum computation* (2019)].

In [2]:
# Build a small HGP code
h = np.loadtxt(
    "../parity_check_matrices/n=12_dv=3_dc=4_dist=6.txt",
    dtype=int,
)
code = HgpCode(h, h)
print("hz shape:", code.hz.shape, " hx shape:", code.hx.shape)
print("lz shape:", code.lz.shape, " lx shape:", code.lx.shape)
code.verify_css_logicals()


hz shape: (108, 225)  hx shape: (108, 225)
lz shape: (9, 225)  lx shape: (9, 225)


{'css_condition': True,
 'lz_commutes_with_X': True,
 'lx_commutes_with_Z': True,
 'rank_hz': 108,
 'rank_hx': 108,
 'rank_lz': 9,
 'rank_lx': 9,
 'k_expected': 9,
 'lz_independent_mod_Z_stabilizers': True,
 'lx_independent_mod_X_stabilizers': True,
 'rank_hz_plus_lz': 117,
 'rank_hx_plus_lx': 117,
 'dim_ker_hz': 117,
 'dim_ker_hx': 117,
 'hx_plus_lx_spans_ker_hz': True,
 'hz_plus_lz_spans_ker_hx': True,
 'same_logicals_ZX_anticommute': True,
 'different_logicals_ZX_commute': True,
 'all_tests_passed': True}

**Canonical logical codewords.** For the HGP code, logical operators are written in the canonical form (see arXiv:2204.10812):
$$L^z_{ik_1+j} = e_i \otimes L^1_j, \quad L^x_{ik_1+j} = L^2_i \otimes e_j$$
where $L^1_j$ ($L^2_i$) is the $j$-th ($i$-th) logical operator of the 1st (2nd) classical code.


In [3]:
# Print canonical logical codewords for the HGP code
for i in range(code.lz.shape[0]):
    print(f'Z{i} :', np.where(code.lz[i, :] == 1)[0], f', X{i} :', np.where(code.lx[i, :] == 1)[0])

Z0 : [111 113 114 115 116 117] , X0 : [ 45  69  81  93 105 117]
Z1 : [108 109 113 114 116 118] , X1 : [ 46  70  82  94 106 118]
Z2 : [110 112 113 114 116 119] , X2 : [ 47  71  83  95 107 119]
Z3 : [123 125 126 127 128 129] , X3 : [  9  21  69  81 105 129]
Z4 : [120 121 125 126 128 130] , X4 : [ 10  22  70  82 106 130]
Z5 : [122 124 125 126 128 131] , X5 : [ 11  23  71  83 107 131]
Z6 : [135 137 138 139 140 141] , X6 : [ 33  57  69  81 105 141]
Z7 : [132 133 137 138 140 142] , X7 : [ 34  58  70  82 106 142]
Z8 : [134 136 137 138 140 143] , X8 : [ 35  59  71  83 107 143]


<a id='qlp-code'></a>

### Quasi-cyclic Lifted Product (QLP) code parameters

Reference: Q. Xu et al., arXiv:2308.08648 (see Table below)

- `b`: base matrix of monomial powers for the polynomial entries
- `lift_size`: circulant size used to lift each monomial entry
- `QlpCode(b, b, lift_size)`: builds a QLP code from two base matrices

Example parameters used in arXiv:2308.08648 are shown below.

| lift_size | [[n, k, d]] | b |
|:---:|:---:|:---:|
| 16 | [[544, 80, $\leq$ 12]] | $\begin{bmatrix} 0 & 0 & 0 & 0 & 0 \\ 0 & 2 & 4 & 7 & 11 \\ 0 & 3 & 10 & 14 & 15 \end{bmatrix}$ |
| 21 | [[714, 100, $\leq$ 16]] | $\begin{bmatrix} 0 & 0 & 0 & 0 & 0 \\ 0 & 4 & 5 & 7 & 17 \\ 0 & 14 & 18 & 12 & 11 \end{bmatrix}$ |
| 30 | [[1020, 136, $\leq$ 20]] | $\begin{bmatrix} 0 & 0 & 0 & 0 & 0 \\ 0 & 2 & 14 & 24 & 25 \\ 0 & 16 & 11 & 14 & 13 \end{bmatrix}$ |
| 42 | [[1428, 184, $\leq$ 24]] | $\begin{bmatrix} 0 & 0 & 0 & 0 & 0 \\ 0 & 6 & 7 & 9 & 30 \\ 0 & 40 & 15 & 31 & 35 \end{bmatrix}$ |



In [4]:
# Build a small QLP code
lift_size = 16
b = np.array(
    [
        [0, 0, 0, 0, 0],
        [0, 2, 4, 7, 11],
        [0, 3, 10, 14, 15],
    ]
)
code = QlpCode(b, b, lift_size)
print("hz shape:", code.hz.shape, " hx shape:", code.hx.shape)
print("lz shape:", code.lz.shape, " lx shape:", code.lx.shape)
code.verify_css_logicals()


hz shape: (240, 544)  hx shape: (240, 544)
lz shape: (80, 544)  lx shape: (80, 544)


{'css_condition': True,
 'lz_commutes_with_X': True,
 'lx_commutes_with_Z': True,
 'rank_hz': 232,
 'rank_hx': 232,
 'rank_lz': 80,
 'rank_lx': 80,
 'k_expected': 80,
 'lz_independent_mod_Z_stabilizers': True,
 'lx_independent_mod_X_stabilizers': True,
 'rank_hz_plus_lz': 312,
 'rank_hx_plus_lx': 312,
 'dim_ker_hz': 312,
 'dim_ker_hx': 312,
 'hx_plus_lx_spans_ker_hz': True,
 'hz_plus_lz_spans_ker_hx': True,
 'same_logicals_ZX_anticommute': True,
 'different_logicals_ZX_commute': True,
 'all_tests_passed': True}

<a id='bpc-code'></a>

### Balanced product cyclic (BPC) code parameters

Reference: R. Tiew et al., arXiv:2411.03302

- `p1`, `p2`: polynomial powers defining the two circulant blocks
- `lift_size`: circulant size of each block
- `factor`: size of the cyclic subgroup factored out

To match arXiv:2411.03302, use `p2` as the transpose convention: each entry is `lift_size` minus the powers listed in the paper, as shown in the table below. Use **lift_size = 3q** and **factor = 3**.

| lift_size (=3q) | [[n, k, d]] | 2q | p1 | p2 |
|:---:|:---:|:---:|:---:|:---:|
| 6 | [[36, 8, 4]] | 4 | [0, 1, 2] | [0, 4, 5] |
| 9 | [[54, 8, 4]] | 6 | [0, 1, 2] | [0, 7, 8] |
| 12 | [[72, 8, 8]] | 8 | [0, 1, 5] | [0, 4, 11] |
| 15 | [[90, 8, 10]] | 10 | [0, 1, 5] | [0, 8, 13] |
| 18 | [[108, 8, 8]] | 12 | [0, 1, 5] | [0, 16, 17] |
| 21 | [[126, 8, 10]] | 14 | [0, 1, 5] | [0, 14, 19] |
| 24 | [[144, 8, 12]] | 16 | [0, 1, 5] | [0, 13, 23] |
| 27 | [[162, 8, 12]] | 18 | [0, 1, 5] | [0, 16, 26] |
| 30 | [[180, 8, 16]] | 20 | [0, 1, 5] | [0, 13, 29] |


In [5]:
# Build a small BPC code
lift_size, factor = 15, 3
p1 = [0, 1, 5]
p2 = [0, 8, 13]
# p2 uses lift_size minus values compared to arXiv:2411.03302 (transpose convention).
code = BpcCode(p1, p2, lift_size, factor, canonical_basis='Z')
print("hz shape:", code.hz.shape, " hx shape:", code.hx.shape)
print("lz shape:", code.lz.shape, " lx shape:", code.lx.shape)
code.verify_css_logicals()


hz shape: (45, 90)  hx shape: (45, 90)
lz shape: (8, 90)  lx shape: (8, 90)


{'css_condition': True,
 'lz_commutes_with_X': True,
 'lx_commutes_with_Z': True,
 'rank_hz': 41,
 'rank_hx': 41,
 'rank_lz': 8,
 'rank_lx': 8,
 'k_expected': 8,
 'lz_independent_mod_Z_stabilizers': True,
 'lx_independent_mod_X_stabilizers': True,
 'rank_hz_plus_lz': 49,
 'rank_hx_plus_lx': 49,
 'dim_ker_hz': 49,
 'dim_ker_hx': 49,
 'hx_plus_lx_spans_ker_hz': True,
 'hz_plus_lz_spans_ker_hx': True,
 'same_logicals_ZX_anticommute': True,
 'different_logicals_ZX_commute': True,
 'all_tests_passed': True}

**Canonical logical codewords.** For the BPC code with odd $q$, logical operators are written in the canonical form, such that codewords of canonical\_basis are of weight $2q$. This is modified from Eq. 30 of arXiv:2411.03302 so that `same_logicals\_ZX\_anticommute' and 'different\_logicals\_ZX\_commute' are True. 


In [6]:
# Print canonical logical codewords for the BPC code
for i in range(code.lz.shape[0]):
    print(f'Z{i} :', np.where(code.lz[i, :] == 1)[0], f', X{i} :', np.where(code.lx[i, :] == 1)[0])


Z0 : [45 46 48 49 51 52 54 55 57 58] , X0 : [45 48 51 54 57 75 78 81 84 87]
Z1 : [46 47 49 50 52 53 55 56 58 59] , X1 : [45 46 48 49 51 52 54 55 57 58 75 76 78 79 81 82 84 85 87 88]
Z2 : [60 61 63 64 66 67 69 70 72 73] , X2 : [60 63 66 69 72 75 78 81 84 87]
Z3 : [61 62 64 65 67 68 70 71 73 74] , X3 : [60 61 63 64 66 67 69 70 72 73 75 76 78 79 81 82 84 85 87 88]
Z4 : [ 0  3  6  9 12 15 18 21 24 27] , X4 : [ 0  2  3  5  6  8  9 11 12 14]
Z5 : [ 1  4  7 10 13 16 19 22 25 28] , X5 : [ 1  2  4  5  7  8 10 11 13 14]
Z6 : [15 18 21 24 27 30 33 36 39 42] , X6 : [ 0  2  3  5  6  8  9 11 12 14 15 17 18 20 21 23 24 26 27 29]
Z7 : [16 19 22 25 28 31 34 37 40 43] , X7 : [ 1  2  4  5  7  8 10 11 13 14 16 17 19 20 22 23 25 26 28 29]


<a id='lcs-code'></a>

### Lift-connected Surface (LCS) code parameters

Reference: J. Old et al., arXiv:2401.02911

- `lift_size`: circulant lift size `L`.
- `length`: number of columns in the base matrix (`length = l + 1`).
- `LcsCode(lift_size, length)`: builds an LCS code from the fixed diagonal pattern with polynomial entries `I` and `I+P`.

In [7]:
# Build a small LCS code
lift_size = 5
length = 3  # length = l + 1

code = LcsCode(lift_size, length)
print("base matrix b:")
print(code.b)
print("hz shape:", code.hz.shape, " hx shape:", code.hx.shape)
print("lz shape:", code.lz.shape, " lx shape:", code.lx.shape)
code.verify_css_logicals()


base matrix b:
[[list([0]) list([0, 1]) list([])]
 [list([]) list([0]) list([0, 1])]]
hz shape: (30, 65)  hx shape: (30, 65)
lz shape: (5, 65)  lx shape: (5, 65)


{'css_condition': True,
 'lz_commutes_with_X': True,
 'lx_commutes_with_Z': True,
 'rank_hz': 30,
 'rank_hx': 30,
 'rank_lz': 5,
 'rank_lx': 5,
 'k_expected': 5,
 'lz_independent_mod_Z_stabilizers': True,
 'lx_independent_mod_X_stabilizers': True,
 'rank_hz_plus_lz': 35,
 'rank_hx_plus_lx': 35,
 'dim_ker_hz': 35,
 'dim_ker_hx': 35,
 'hx_plus_lx_spans_ker_hz': True,
 'hz_plus_lz_spans_ker_hx': True,
 'same_logicals_ZX_anticommute': True,
 'different_logicals_ZX_commute': True,
 'all_tests_passed': True}

<a id='bb-code'></a>

### Bivariate Bicycle (BB) code parameters

Reference: S. Bravyi et al., arXiv:2308.07915 (Table 3 values shown below)

- `l, m`: circulant dimensions in the bivariate construction.
- `A_x_pows`, `A_y_pows`: powers for x-terms and y-terms in `A(x, y)`.
- `B_x_pows`, `B_y_pows`: powers for x-terms and y-terms in `B(x, y)`.
- `BbCode(l, m, A_x_pows, A_y_pows, B_x_pows, B_y_pows)`: builds the BB code.

| [[n, k, d]] | l, m | A_x_pows | A_y_pows | B_x_pows | B_y_pows |
|:---:|:---:|:---:|:---:|:---:|:---:|
| [[72, 12, 6]] | 6, 6 | [3] | [1, 2] | [1, 2] | [3] |
| [[90, 8, 10]] | 15, 3 | [9] | [1, 2] | [2, 7] | [0] |
| [[108, 8, 10]] | 9, 6 | [3] | [1, 2] | [1, 2] | [3] |
| [[144, 12, 12]] | 12, 6 | [3] | [1, 2] | [1, 2] | [3] |
| [[288, 12, 18]] | 12, 12 | [3] | [2, 7] | [1, 2] | [3] |
| [[360, 12, $\leq$ 24]] | 30, 6 | [9] | [1, 2] | [25, 26] | [3] |
| [[756, 16, $\leq$ 34]] | 21, 18 | [3] | [10, 17] | [3, 19] | [5] |


In [8]:
# Build a small BB code
l, m = 12, 6
A_x_pows = [3]
A_y_pows = [1, 2]
B_x_pows = [1, 2]
B_y_pows = [3]

code = BbCode(l, m, A_x_pows, A_y_pows, B_x_pows, B_y_pows)
print("hz shape:", code.hz.shape, " hx shape:", code.hx.shape)
print("lz shape:", code.lz.shape, " lx shape:", code.lx.shape)
code.verify_css_logicals()


hz shape: (72, 144)  hx shape: (72, 144)
lz shape: (12, 144)  lx shape: (12, 144)


{'css_condition': True,
 'lz_commutes_with_X': True,
 'lx_commutes_with_Z': True,
 'rank_hz': 66,
 'rank_hx': 66,
 'rank_lz': 12,
 'rank_lx': 12,
 'k_expected': 12,
 'lz_independent_mod_Z_stabilizers': True,
 'lx_independent_mod_X_stabilizers': True,
 'rank_hz_plus_lz': 78,
 'rank_hx_plus_lx': 78,
 'dim_ker_hz': 78,
 'dim_ker_hx': 78,
 'hx_plus_lx_spans_ker_hz': True,
 'hz_plus_lz_spans_ker_hx': True,
 'same_logicals_ZX_anticommute': True,
 'different_logicals_ZX_commute': True,
 'all_tests_passed': True}

<a id='mitten-code'></a>

### Mitten code parameters

Reference: A. Bhardwaj et al., arXiv:2607.28795

Mitten codes are $1 \times 2$ lifted products over the group algebra $\mathbb{F}_2[G]$ of a **non-abelian** finite group $G$.

- `A`, `B`: $1 \times 2$ ring matrices $[a_0 \ a_1]$, $[b_0 \ b_1]$ over $\mathbb{F}_2[G]$ — each entry a tuple of 3 group-element indices, with the identity (index 0) in $a_1$, $b_1$ and $L(a_1)$, $R(b_1)$ full rank.
- `group`: the group $G$ as a `quits.gf2_util.MultTableGroup` — build it with `cyclic`, `symmetric`, `dihedral`, `alternating`, `direct_product`, or from explicit permutation generators via `MultTableGroup.from_generators`.
- `MittenCode(A, B, group)`: builds the mitten code.
- `sample_mitten_base_matrices(group, rng)`: draws random base matrices until they form a valid mitten pair.

Every mitten code has $n = 5|G|$ data qubits and $k = |G|$ logical qubits (rate exactly 20%), check weight 9, and qubit weight 6 on the four "finger" data blocks / 12 on the "thumb" block. Published instances (Table 1 of the paper):

| [[n, k, d]] | group $G$ |
|:---:|:---:|
| [[150, 30, 10]] | $C_5 \times S_3$ |
| [[200, 40, 12]] | $C_4 \times D_{10}$ |
| [[300, 60, 14]] | $C_{10} \times S_3$ |
| [[500, 100, 16]] | $C_5 \rtimes C_{20}$ |
| [[540, 108, 18]] | $C_9 \rtimes C_{12}$ |
| [[630, 126, $\leq$ 20]] | $C_7 \rtimes C_{18}$ |
| [[780, 156, $\leq$ 22]] | $C_{13} \times A_4$ |
| [[975, 195, $\leq$ 24]] | $C_{13} \rtimes C_{15}$ |

The example below builds a [[500, 100]] code over $C_5 \rtimes C_{20} = \langle c, y \mid c^5 = y^{20} = e,\ y c y^{-1} = c^3 \rangle$, the group of the paper's [[500, 100, 16]] instance, realized as a permutation group. The base matrices are randomly sampled, which fixes $n$, $k$, and all the weights above but not the distance — reaching $d = 16$ requires the optimized pair from the paper's code-construction appendix.

In [9]:
# Build a mitten code over C5 : C20 = <c, y | c^5 = y^20 = e, y c y^-1 = c^3>,
# as permutations on 25 points: c 5-cycles {0..4}; y multiplies {0..4} by 3
# (mod 5), realizing y c y^-1 = c^3, and 20-cycles {5..24}, making its order 20.
c = tuple([(i + 1) % 5 for i in range(5)] + list(range(5, 25)))
y = tuple([(3 * i) % 5 for i in range(5)] + [5 + (j + 1) % 20 for j in range(20)])
G = MultTableGroup.from_generators([c, y], "C5 : C20")
print(G)

A, B = sample_mitten_base_matrices(G, np.random.default_rng(1))
print("A =", A, " B =", B)

code = MittenCode(A, B, G)
print("hz shape:", code.hz.shape, " hx shape:", code.hx.shape)
print("lz shape:", code.lz.shape, " lx shape:", code.lx.shape)
code.verify_css_logicals()


MultTableGroup(structure='C5 : C20', n=100, is_abelian=False)
A = [[(46, 50, 75), (0, 15, 82)]]  B = [[(24, 30, 86), (0, 26, 82)]]
hz shape: (200, 500)  hx shape: (200, 500)
lz shape: (100, 500)  lx shape: (100, 500)


{'css_condition': True,
 'lz_commutes_with_X': True,
 'lx_commutes_with_Z': True,
 'rank_hz': 200,
 'rank_hx': 200,
 'rank_lz': 100,
 'rank_lx': 100,
 'k_expected': 100,
 'lz_independent_mod_Z_stabilizers': True,
 'lx_independent_mod_X_stabilizers': True,
 'rank_hz_plus_lz': 300,
 'rank_hx_plus_lx': 300,
 'dim_ker_hz': 300,
 'dim_ker_hx': 300,
 'hx_plus_lx_spans_ker_hz': True,
 'hz_plus_lz_spans_ker_hx': True,
 'same_logicals_ZX_anticommute': True,
 'different_logicals_ZX_commute': True,
 'all_tests_passed': True}

**Canonical logical codewords.** The full-rank condition on $a_1$, $b_1$ equips every mitten code with a canonical logical basis indexed by the group itself (Theorem 1 of arXiv:2607.28795): on the five data blocks,
$$\bar{X}_g = (\mathbb{1}_g,\, u_g,\, 0,\, 0,\, 0), \qquad \bar{Z}_g = (\mathbb{1}_g,\, 0,\, v_g,\, 0,\, 0),$$
so that `same_logicals_ZX_anticommute` and `different_logicals_ZX_commute` are True, with each conjugate pair $(\bar{X}_g, \bar{Z}_g)$ overlapping on exactly one qubit.

In [10]:
# Print canonical logical codewords for the first few logical qubits
for i in range(3):
    print(f'Z{i} :', np.where(code.lz[i, :] == 1)[0], f', X{i} :', np.where(code.lx[i, :] == 1)[0])


Z0 : [  0 200 204 215 217 218 219 220 221 226 229 248 249 251 252 271 274 279
 295 297] , X0 : [  0 116 117 118 119 120 135 137 138 140 142 143 144 145 147 148 149 166
 167 168 169 170 172 174 190 192 194 195 197 198]
Z1 : [  1 200 201 215 216 218 219 221 222 225 227 245 249 252 253 270 272 275
 296 298] , X1 : [  1 115 117 118 119 121 136 138 139 140 141 143 144 145 146 148 149 165
 167 168 169 170 171 173 190 191 193 196 198 199]
Z2 : [  2 201 202 215 216 217 219 222 223 226 228 245 246 253 254 271 273 276
 297 299] , X2 : [  2 115 116 118 119 122 135 137 139 140 141 142 144 145 146 147 149 165
 166 168 169 171 172 174 191 192 194 195 197 199]
